## TRAIN.CSV- EDA AND PREPROCESSING

In [ ]:
from datasets import load_dataset

train_data = load_dataset("google/civil_comments", split="train")
train_data.to_csv("civil_comments_train.csv")
sample = train_data.shuffle(seed=42).select(range(50_000))
sample.to_csv("civil_comments_sample.csv")
import os
print(os.getcwd())
from pathlib import Path

csv_path = Path("civil_comments_train.csv").resolve()
print("Saved:", csv_path)
print("Exists:", csv_path.exists())

if csv_path.exists():
    print("Size in MB:", round(csv_path.stat().st_size / (1024 * 1024), 1))

## Planned model labels

### Civil Comments model
This model will predict:
- Broad toxicity
- Obscene/profane language
- Insults
- Threats
- Identity attacks
- Sexually explicit content

### Davidson model
The `class` column means:
- `0` = hate speech
- `1` = offensive language
- `2` = neither

For the broad abuse prediction, classes `0` and `1` will count as abusive, and class `2` as non-abusive. Davidson does not have separate labels for threats, insults, or profanity.

### Ensemble
The two models will contribute to a shared broad abuse prediction. The Civil Comments model will provide the more specific category predictions.

# EDA-CIVIL_DATASET


In [ ]:
import pandas as pd

civil_df = pd.read_csv("civil_comments_sample.csv")
davidson_df = pd.read_csv("train.csv")

In [ ]:
civil_df.shape

In [ ]:
civil_df.describe()

In [ ]:
civil_df.head()

In [ ]:
civil_df.info()

In [ ]:
civil_df.isnull().sum()

In [ ]:
civil_df.duplicated().sum()

In [ ]:
civil_df["text"].duplicated().sum()

In [ ]:
label_columns = [
    "toxicity", "severe_toxicity", "obscene", "threat",
    "insult", "identity_attack", "sexual_explicit"
]

repeated_text = civil_df[civil_df["text"].duplicated(keep=False)]

display(
    repeated_text[["text"] + label_columns]
    .sort_values("text")
    .head(50000)
)

In [ ]:
label_columns = [
    "toxicity", "severe_toxicity", "obscene", "threat",
    "insult", "identity_attack", "sexual_explicit"
]

# Make a working copy and remove rows identical across every column
civil_clean = civil_df.drop_duplicates().copy()

print("Rows before:", len(civil_df))
print("Rows after exact-duplicate removal:", len(civil_clean))

# Check whether any repeated text has different labels
label_conflicts = (
    civil_clean.groupby("text")[label_columns]
    .nunique()
    .gt(1)
    .any(axis=1)
)

print("Repeated texts with different label scores:", label_conflicts.sum())

In [ ]:
civil_clean = civil_clean.drop_duplicates(subset="text", keep="first").copy()

print("Rows after text deduplication:", len(civil_clean))
print("Repeated texts remaining:", civil_clean["text"].duplicated().sum())

- We found out using `.duplicated().sum()` that some rows were duplicated. First, we checked how many rows there were, and we then dealt with it. 
- There were 129 rows that were duplicated, so we removed those duplicated rows and saved it in another copy. 
- Then we saw that Some duplicated rows after deduplicating were still left that had repeated texts with different label scores, so we removed them also. 

# EDA-DAVISON DATASET

In [ ]:
davidson_df.head()


In [ ]:
davidson_df.shape


In [ ]:
davidson_df.info()


In [ ]:
davidson_df.isnull().sum()


In [ ]:
davidson_df.duplicated().sum()


In [ ]:
davidson_df["class"].value_counts()